In [ ]:
# Import libraries
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired

import re
import json

In [ ]:
sentence_model_name = "sentence-transformers/all-MiniLM-L6-v2" #Also use: BAAI/bge-base-en-v1.5

json_file = "all_comments.json"

In [ ]:
# Load models
sentence_embedding_model = SentenceTransformer(sentence_model_name)

# Experiment with different models

# UMAP
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# Use a better Topic Representation
representation_model = KeyBERTInspired()

In [ ]:
# Load the comments
# One comment = one document

new_data = []

with open(json_file, "r", encoding="utf-8") as f:
    new_data = json.load(f)

print(f"Loaded: {len(new_data)} comments")

# Extract the text
comments = [data['post_comment'] for data in new_data]

In [ ]:
# Perform pre-processing: remove URLs, NaN
def clean_data(text):

    # Remove URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove the \n \r
    text = " ".join(text.split())

    # Handle NaN
    if isinstance(text, float):
        text = ""

    # Remove extra space
    text = re.sub(r'\s+', ' ', text)

    return text

cleaned_comments = [clean_data(comment) for comment in comments]
print(cleaned_comments[:100])

In [ ]:
# Generate the embeddings
embeddings = sentence_embedding_model.encode(
    cleaned_comments,
    show_progress_bar = True
)

In [ ]:
# Create the BERTopic model

# Baseline
topic_model = BERTopic(
    embedding_model=sentence_embedding_model,
    verbose=True
)


# Using UMAP
#topic_model = BERTopic(
#    embedding_model=sentence_embedding_model,
#    umap_model=umap_model,
#    random_state=42
#)

# Using HDBSCAN
#topic_model = BERTopic(
#    embedding_model=sentence_embedding_model,
#    umap_model=umap_model,
#    hdbscan_model=hdbscan_model
#)

# Using Representation model
#topic_model = BERTopic(
#    embedding_model=sentence_embedding_model,
#    representation_model=representation_model
#)

In [ ]:
topics, probabilities = topic_model.fit_transform(
    cleaned_comments,
    embeddings
)

In [ ]:
# Inspect the topics
topic_info = topic_model.get_topic_info()
print(topic_info)



In [ ]:
# Examine individual topics
topic_model.get_topic(0)

In [ ]:
# Look at representative comments
topic_model.get_representative_docs(0)

In [ ]:
# Visualize the topics
fig = topic_model.visualize_topics()
fig.show()



In [ ]:
# Visualize the topic hierarchy
fig = topic_model.visualize_hierarchy()
fig.show()

In [ ]:
# Look at the topic frequency
topic_model.visualize_barchart()

In [ ]:
# Perform topic reduction if needed
topic_model.reduce_topics(
    cleaned_comments,
    nr_topics=20
)

In [ ]:
# Keep clustering and labeling separate

In [ ]:
# Topic validations

# Export topic assignment for every reddit comment